# Đánh giá Offline (Offline Evaluation)

Notebook này giúp bạn tự chấm điểm (tính toán PSNR, SSIM, LPIPS và Final Score) cho các ảnh render từ mô hình so với tập ảnh Ground Truth (ví dụ: tập test của `public_set`).

Công thức của BTC:
`Score = 0.4 * (1 - LPIPS) + 0.3 * SSIM + 0.3 * PSNR_norm`

*(Giả định PSNR_max = 40 để chuẩn hóa PSNR_norm)*

In [ ]:
!pip install -q lpips scikit-image

In [ ]:
import os
import cv2
import numpy as np
import torch
import lpips
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from tqdm.auto import tqdm

def load_image_for_skimage(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def load_image_for_lpips(path):
    # LPIPS expects input in range [-1, 1]
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = (img / 255.0) * 2.0 - 1.0
    # Convert to NCHW tensor
    tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0)
    return tensor

In [ ]:
# ---------------------------------------------------------
# CẤU HÌNH ĐƯỜNG DẪN
# ---------------------------------------------------------
# Thay đổi đường dẫn này trỏ tới thư mục ảnh Ground Truth
GT_DIR = '/kaggle/input/datasets/dangthtai/bts-digital-twin-dataset/phase1/public_set/HCM0204/test/images'

# Thay đổi đường dẫn này trỏ tới thư mục ảnh Render (Infer) của bạn
RENDER_DIR = '/kaggle/working/output/HCM0204/renders'

# PSNR Max do BTC quy định (giả định 40)
PSNR_MAX = 40.0

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loss_fn_vgg = lpips.LPIPS(net='vgg').to(device)

psnr_list = []
ssim_list = []
lpips_list = []

if not os.path.exists(GT_DIR):
    print(f"Thư mục GT không tồn tại: {GT_DIR}")
elif not os.path.exists(RENDER_DIR):
    print(f"Thư mục Render không tồn tại: {RENDER_DIR}")
else:
    filenames = sorted([f for f in os.listdir(RENDER_DIR) if f.endswith(('.png', '.jpg'))])
    
    for fname in tqdm(filenames, desc="Evaluating"):
        gt_path = os.path.join(GT_DIR, fname)
        render_path = os.path.join(RENDER_DIR, fname)
        
        if not os.path.exists(gt_path):
            print(f"Không tìm thấy ảnh GT tương ứng cho {fname}")
            continue
            
        # Tính PSNR & SSIM
        gt_img = load_image_for_skimage(gt_path)
        render_img = load_image_for_skimage(render_path)
        
        # Đảm bảo cùng kích thước
        if gt_img.shape != render_img.shape:
            render_img = cv2.resize(render_img, (gt_img.shape[1], gt_img.shape[0]))
            
        val_psnr = psnr(gt_img, render_img, data_range=255)
        val_ssim = ssim(gt_img, render_img, data_range=255, channel_axis=-1)
        
        # Tính LPIPS
        gt_tensor = load_image_for_lpips(gt_path).to(device)
        render_tensor = load_image_for_lpips(render_path).to(device)
        
        with torch.no_grad():
            val_lpips = loss_fn_vgg(gt_tensor, render_tensor).item()
            
        psnr_list.append(val_psnr)
        ssim_list.append(val_ssim)
        lpips_list.append(val_lpips)
        
    if len(psnr_list) > 0:
        avg_psnr = np.mean(psnr_list)
        avg_ssim = np.mean(ssim_list)
        avg_lpips = np.mean(lpips_list)
        
        # Normalization (Clamp)
        psnr_norm = max(0.0, min(avg_psnr / PSNR_MAX, 1.0))
        
        score = 0.4 * (1 - avg_lpips) + 0.3 * avg_ssim + 0.3 * psnr_norm
        score_100 = score * 100
        
        print("\n" + "="*40)
        print(f"KẾT QUẢ ĐÁNH GIÁ TRÊN {len(psnr_list)} ẢNH:")
        print("="*40)
        print(f"PSNR  : {avg_psnr:.4f} dB")
        print(f"SSIM  : {avg_ssim:.4f}")
        print(f"LPIPS : {avg_lpips:.4f}")
        print("-"*40)
        print(f"FINAL SCORE: {score_100:.2f} / 100")
        print("="*40)
    else:
        print("Không có ảnh hợp lệ nào được đánh giá!")